# Projections and Least Squares Geometry

This notebook explains the column space projection, visualizes it in two and three dimensions, and shows how QR-based least squares compares to the normal equations for noisy data.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from linalg_with_python.least_squares import least_squares_normal_eq, least_squares_qr

FIGURES_DIR = Path.cwd() / "assets" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## Column space projections

We project a target vector onto the span of a matrix's columns, then repeat the experiment in three dimensions so you can see the resulting plane.


In [ ]:

A = np.array([[1.0, 0.3], [0.5, 1.0]], dtype=np.float64)
b = np.array([2.0, 1.5], dtype=np.float64)
result = least_squares_qr(A, b)
projection = A @ result.x

fig, ax = plt.subplots(figsize=(5, 5))
for idx, vec in enumerate(A.T):
    ax.arrow(0, 0, vec[0], vec[1], head_width=0.03, length_includes_head=True, color=f"C{idx}", label=f"column {idx + 1}")
ax.arrow(0, 0, b[0], b[1], head_width=0.05, length_includes_head=True, color="black", label="target b")
ax.arrow(0, 0, projection[0], projection[1], head_width=0.05, length_includes_head=True, color="tab:orange", linestyle="--", label="projection")
ax.set_xlim(-0.5, 2.5)
ax.set_ylim(-0.5, 2.5)
ax.set_aspect("equal", "box")
ax.set_title("Projection of b onto the column space")
ax.grid(True, linestyle=":")
ax.legend()
path = FIGURES_DIR / "projection_2d.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved 2D projection figure to {path}")


### Projection onto a plane in R^3

We now project a vector onto a two-dimensional subspace inside R^3, visualized as a faint plane.


In [ ]:

plane = np.array([[1.0, 0.0], [0.5, 1.0], [0.0, 0.3]], dtype=np.float64)
vector = np.array([1.0, 1.5, 0.2], dtype=np.float64)
result3d = least_squares_qr(plane, vector)
projection3d = plane @ result3d.x

U = np.linspace(-1.5, 1.5, 20)
V = np.linspace(-1.5, 1.5, 20)
U, V = np.meshgrid(U, V)
plane_pts = (
    plane[:, 0][:, None, None] * U[None, ...]
    + plane[:, 1][:, None, None] * V[None, ...]
)

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(projection="3d")
ax.plot_surface(plane_pts[0], plane_pts[1], plane_pts[2], alpha=0.3, color="tab:green")
ax.plot([0, vector[0]], [0, vector[1]], [0, vector[2]], color="black", linewidth=2, label="vector")
ax.plot([0, projection3d[0]], [0, projection3d[1]], [0, projection3d[2]], color="tab:orange", linestyle="--", linewidth=2, label="projection")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title("Projection onto a 2D subspace in R^3")
ax.legend()
path = FIGURES_DIR / "projection_3d.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved 3D projection figure to {path}")


## Line fitting from noisy points

We reuse the QR least squares line fitting demo to show how the algorithm returns a smooth estimate despite noise.


In [ ]:

rng = np.random.default_rng(0)
x = np.linspace(-2.0, 2.0, 40)
y = 1.0 + 2.0 * x + rng.normal(scale=0.5, size=x.shape)
A_ls = np.column_stack([np.ones_like(x), x])
result_ls = least_squares_qr(A_ls, y)
y_hat = A_ls @ result_ls.x

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(x, y, label="noisy data", color="tab:blue", alpha=0.7)
ax.plot(x, y_hat, label="QR least squares fit", color="tab:orange")
ax.set_title("Line fitting via QR least squares")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid(True, linestyle=":")
ax.legend()
path = FIGURES_DIR / "line_fit_qr.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved line fit figure to {path}")
print(f"Residual norm: {result_ls.residual_norm:.3e}")


## Normal equations vs QR

We perturb the right-hand side and compare how much QR and the normal equations deviate from the ground truth.


In [ ]:

rng = np.random.default_rng(42)
true = np.array([1.0, 2.0], dtype=np.float64)
A_cmp = A_ls
b_cmp = A_cmp @ true
noise_levels = np.geomspace(1e-6, 1e-2, 8)
errors_normal = []
errors_qr = []
for eps in noise_levels:
    noise = rng.normal(scale=eps, size=b_cmp.shape)
    normal_result = least_squares_normal_eq(A_cmp, b_cmp + noise)
    qr_result = least_squares_qr(A_cmp, b_cmp + noise)
    errors_normal.append(np.linalg.norm(normal_result.x - true))
    errors_qr.append(np.linalg.norm(qr_result.x - true))

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(noise_levels, errors_normal, marker="o", label="Normal eq")
ax.loglog(noise_levels, errors_qr, marker="s", label="QR")
ax.set_title("Solution error vs noise level")
ax.set_xlabel("noise magnitude")
ax.set_ylabel("||x_true - x_est||₂")
ax.grid(True, which="both", linestyle=":", linewidth=0.5)
ax.legend()
path = FIGURES_DIR / "normal_vs_qr_residuals.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved comparison figure to {path}")


### Conclusion

QR least squares avoids squaring the condition number, so it remains stable whenever the normal equations start blowing up. The figures above illustrate projection geometry and the numerical benefit of QR.


In [ ]:
from linalg_with_python.checks import assert_close, is_upper_triangular
from linalg_with_python.decompositions import qr_gram_schmidt
import numpy as np
x = np.linspace(-2.0, 2.0, 40)
A = np.column_stack([np.ones_like(x), x])
qr = qr_gram_schmidt(A, method='modified')
print('Upper triangular R?', is_upper_triangular(qr.R, tol=1e-12))
assert_close('R matches upper triangle', qr.R, np.triu(qr.R), tol=1e-12)
print('R is numerically upper triangular.')